# 32 — Data Transformation for Regression (Objective 3)

**Objective.** Create the model-ready regression table, make the train/test split before any feature-engineering decisions, and save the feature specification.

**Input:** `data/preprocessed/warehouse_preprocessed.csv`.

**Output:** `data/processed/regression_model_input.csv`, `feature_engine/model_features.csv`, and `feature_engine/feature_spec.md`.

This notebook documents the target scale, split placement and encoding choices that all modelling steps must follow.

## 0. Setup

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "common.py").exists())
sys.path.insert(0, str(ROOT))
from src.common import *
set_style()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
from sklearn.model_selection import train_test_split

p = obj_paths(3)
for folder in [p["processed"], p["feature_engine"], p["train_eval"], p["model"]]:
    folder.mkdir(parents=True, exist_ok=True)

## 1. Target representation

The choice of target scale affects how model errors are reported and how recommendations are communicated to a business audience. Keeping shipment weight in recorded tons means that MAE and RMSE are directly readable in the same units as the business question.

In [ ]:
df = load_preprocessed()
target_summary = df["product_wg_ton"].describe().to_frame("product_wg_ton")
target_shape = pd.DataFrame({
    "measure": ["skew", "kurtosis", "min", "median", "max"],
    "value": [df["product_wg_ton"].skew(), df["product_wg_ton"].kurt(), df["product_wg_ton"].min(), df["product_wg_ton"].median(), df["product_wg_ton"].max()],
})
display(target_summary.round(3))
display(target_shape.round(3))

> **Interpretation.**
>
> - The target has a skew of 0.33 and kurtosis of -0.50, both very close to zero.
> - The range of 2,065 to 55,151 tons is wide but without extreme outliers that would force a transformation.
> - Mean (22,103) and median (22,101) are almost identical, confirming near-symmetry.
> - Keeping the target in recorded tons means that prediction errors are directly readable as tonnes shipped.

> **Decision — target transformation.**
>
> - Skew is mild, not a strong reason for a log or square-root transform.
> - The target is a business quantity: tons shipped in the last three months.
> - Keeping tons makes MAE, RMSE and recommendations directly readable.
> - Rejected: log target and square-root target.

## 2. Train/test split

Fixing the train/test split here — before any feature engineering decisions — prevents data leakage: no information from the test rows can influence encoding choices or derived features made in later steps. This is one of the most important structural decisions in the regression pipeline.

In [ ]:
train_idx, test_idx = train_test_split(df.index, test_size=0.20, random_state=RANDOM_STATE)
split = pd.Series("train", index=df.index)
split.loc[test_idx] = "test"

split_check = pd.DataFrame({
    "full": df["product_wg_ton"].describe(),
    "train": df.loc[train_idx, "product_wg_ton"].describe(),
    "test": df.loc[test_idx, "product_wg_ton"].describe(),
})
display(split_check.round(2))

> **Interpretation.**
>
> - The training mean (22,109) and test mean (22,076) are within 0.1% of each other.
> - All quartile ranges align well across the full dataset, training set and test set.
> - This confirms that the random split did not accidentally concentrate high or low shipment values in one set.
> - Both sets are therefore representative of the full warehouse population.

> **Decision — train/test split.**
>
> - The training set has 20,000 warehouses and the test set has 5,000.
> - Target mean, median and quartiles are almost identical across full, train and test tables.
> - Feature-engineering decisions in the feature engineering step and model selection in the modelling step therefore use training rows only.
> - Rejected: making the split after feature engineering decisions.

## 3. Encoding and scaling specification

Documenting the encoding and scaling rules before model training ensures that all models in the comparison step see identical input representations. This makes the feature decisions reproducible and easy to audit, which is important for a project that compares multiple model families.

In [ ]:
model = df.copy()
model["certificate_grade"] = model["approved_wh_govt_certificate"].map({"Unrated": 0, "C": 1, "B": 2, "B+": 3, "A": 4, "A+": 5})
model["capacity_size"] = model["WH_capacity_size"].map({"Small": 1, "Mid": 2, "Large": 3})
model["Location_type_Urban"] = model["Location_type"].eq("Urban").astype(int)
model["wh_owner_type_Rented"] = model["wh_owner_type"].eq("Rented").astype(int)
for level in ["East", "South", "West"]:
    model[f"zone_{level}"] = model["zone"].eq(level).astype(int)
for level in ["Zone 1", "Zone 2", "Zone 3", "Zone 4", "Zone 5"]:
    model[f"WH_regional_zone_{level.replace(' ', '_')}"] = model["WH_regional_zone"].eq(level).astype(int)

model_features = [
    "storage_issue_reported_l3m", "wh_breakdown_l3m", "num_refill_req_l3m", "govt_check_l3m", "transport_issue_l1y",
    "wh_est_year", "workers_num", "dist_from_hub", "Competitor_in_mkt", "retail_shop_num", "distributor_num",
    "electric_supply", "temp_reg_mach", "flood_proof", "flood_impacted", "is_unrated_warehouse", "wh_est_year_missing",
    "certificate_grade", "capacity_size", "Location_type_Urban", "wh_owner_type_Rented", "zone_East", "zone_South", "zone_West",
    "WH_regional_zone_Zone_1", "WH_regional_zone_Zone_2", "WH_regional_zone_Zone_3", "WH_regional_zone_Zone_4", "WH_regional_zone_Zone_5",
]
scale_for_linear_svr = [
    "storage_issue_reported_l3m", "wh_breakdown_l3m", "num_refill_req_l3m", "govt_check_l3m", "transport_issue_l1y",
    "wh_est_year", "workers_num", "dist_from_hub", "Competitor_in_mkt", "retail_shop_num", "distributor_num",
    "certificate_grade", "capacity_size",
]

feature_rows = []
for feature in model_features:
    source = feature
    if feature.startswith("zone_"):
        source = "zone"
    if feature.startswith("WH_regional_zone_"):
        source = "WH_regional_zone"
    if feature == "certificate_grade":
        source = "approved_wh_govt_certificate"
    if feature == "capacity_size":
        source = "WH_capacity_size"
    if feature == "Location_type_Urban":
        source = "Location_type"
    if feature == "wh_owner_type_Rented":
        source = "wh_owner_type"
    role = "period measure" if feature in ["storage_issue_reported_l3m", "wh_breakdown_l3m", "num_refill_req_l3m", "govt_check_l3m", "transport_issue_l1y"] else ("recording flag" if feature == "wh_est_year_missing" else "characteristic")
    encoding = "numeric, as recorded" if feature in df.columns else ("ordinal" if feature in ["certificate_grade", "capacity_size"] else "one-hot / 0-1")
    feature_rows.append({
        "feature": feature,
        "source_column": source,
        "role": role,
        "encoding": encoding,
        "scale_for_linear_svr": feature in scale_for_linear_svr,
    })
feature_table = pd.DataFrame(feature_rows)

audit = pd.DataFrame({
    "item": ["rows", "model_features", "nulls_in_model_features", "target_nulls"],
    "value": [len(model), len(model_features), int(model[model_features].isna().sum().sum()), int(model["product_wg_ton"].isna().sum())],
})
display(feature_table)
display(audit)

> **Interpretation.**
>
> - The feature table documents 29 encoded predictors across three role types: period measures, warehouse characteristics and one recording flag.
> - The audit confirms zero null values in any predictor or in the target, so the model-ready table is complete without imputation.
> - Ordinal encoding for certificate grade and capacity size preserves their natural order while keeping them as single columns.
> - The scale flag separates the 13 columns that need standardisation inside linear model pipelines from the 16 that are already on usable numeric scales for tree-based models.

> **Decision — feature types.**
>
> - Certificate and capacity have natural orders, so they are ordinal.
> - Location and ownership have two levels, so each becomes one 0/1 column.
> - Zone and regional zone have no order, so they are one-hot encoded with one reference level left out.
> - Linear, Ridge, Lasso and Linear SVR scale numeric and ordinal columns inside their pipelines.
> - Tree, random forest, AdaBoost, XGBoost and CatBoost use the recorded numeric scale.

## 4. Save outputs

In [ ]:
model_input = model[["Ware_house_ID", "Location_type", "zone", "WH_regional_zone", "product_wg_ton"] + model_features].copy()
model_input.insert(1, "split", split.values)
model_input = model_input[["Ware_house_ID", "split", "Location_type", "zone", "WH_regional_zone", "product_wg_ton"] + model_features]

save_table(model_input, p["processed"] / "regression_model_input.csv", index=False)
save_table(model_input, p["processed"] / "regression_base.csv", index=False)
save_table(feature_table, p["feature_engine"] / "model_features.csv", index=False)

feature_spec = f"""# Objective 3 — regression feature specification

Written by `notebooks/32_data_transformation.ipynb`.

## Rows and target
- 25,000 warehouses, all kept.
- Target: `product_wg_ton`, kept in recorded tons.
- Split: 20,000 train / 5,000 test, `random_state = {RANDOM_STATE}`.

## Features
- {len(model_features)} encoded predictors.
- Period measures are kept because Objective 3 models shipment weight as an operational outcome in the same snapshot.
- Linear and Linear SVR pipelines scale the {len(scale_for_linear_svr)} numeric / ordinal columns on training rows only.
- Tree and boosting models use unscaled inputs.

## Encoding
- Certificate: Unrated 0, C 1, B 2, B+ 3, A 4, A+ 5.
- Capacity: Small 1, Mid 2, Large 3.
- Binary columns: `Location_type_Urban`, `wh_owner_type_Rented`.
- One-hot columns: `zone_East`, `zone_South`, `zone_West`, and `WH_regional_zone_Zone_1` through `_Zone_5`.
"""
(p["feature_engine"] / "feature_spec.md").write_text(feature_spec)
print(feature_spec)

## 5. Checks

In [ ]:
reloaded = pd.read_csv(p["processed"] / "regression_model_input.csv")
assert reloaded.shape == (25_000, 35)
assert reloaded["split"].value_counts().to_dict() == {"train": 20_000, "test": 5_000}
assert reloaded[model_features + ["product_wg_ton"]].isna().sum().sum() == 0
print("checks passed")

---
## Summary

The shipment weight target is kept in recorded tons because its skew (0.33) is mild and business readability matters more than distributional normality for a BBA project. A fixed 80/20 random split is made here so that all feature-engineering and model-selection decisions later use only the 20,000 training rows. The model-ready table contains 25,000 warehouses and 29 encoded predictors with no missing values. Encoding follows a simple logic: ordinal for certificate grade and warehouse capacity, one-hot for zone and regional zone, and binary columns for location type and ownership. The feature engineering step builds on this table using training rows only.